<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 8B: *Fire Ignition Feature Ablation*
##### Version Number: 4.0
---
### Contents  
> *Build Models*\
> *SHAP*\
> *Set Ablation*
---
### Notes
---
### Inputs
---
### Outputs 
---
### User Defined Dependencies

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import json

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from sklearn.model_selection import train_test_split

---

###  Load Data

In [3]:
X_ignition = pd.read_csv('../data/processed/X_ignition.csv')
y_ignition = pd.read_csv('../data/processed/y_ignition.csv').squeeze()  # Load as Series      
details_ignition = pd.read_csv('../data/processed/details_ignition.csv')

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
pal_y = pd.read_csv('../data/processed/pal_y.csv')

best_strategy = pd.read_csv('../data/processed/ignition_best_strategy.csv')

with open('model_parameters_ignition.json', 'r') as f:
    model_parameters = json.load(f)

with open('feature_sets.json', 'r') as f:
    feature_sets = json.load(f)

## Subset

In [4]:
reform = pd.concat([X_ignition,y_ignition], axis=1)
subset = subset_df(reform, 'Target_Ignition_Lag', 500)

y = subset['Target_Ignition_Lag']
X = subset.drop(columns='Target_Ignition_Lag')

In [5]:
X_rf, y_rf = apply_balancing('RF', best_strategy, X, y)
X_xgb, y_xgb = apply_balancing('XGB', best_strategy, X, y)

## Build Models

In [6]:
RF_parameters = model_parameters['Random Forest']
XGB_parameters = model_parameters['XGBoost']

# Build tuned models
ignition_xbg = xgb.XGBClassifier(**XGB_parameters)
ignition_rf = RandomForestClassifier(**RF_parameters)

display(RF_parameters)
display(XGB_parameters)

{'n_estimators': 50,
 'max_depth': 10,
 'min_samples_split': 5,
 'max_features': 'log2',
 'class_weight': 'balanced'}

{'objective': 'multi:softmax',
 'num_class': 3,
 'n_estimators': 150,
 'max_depth': 5,
 'learning_rate': 0.1,
 'verbosity': 0}

In [7]:
models = {
    "XGB": ignition_xbg, 
    "RF": ignition_rf
}

## SHAP

### XGB Ignition SHAP

In [8]:
xgb_top_10 = get_shap(ignition_xbg, X_xgb, y_xgb)
xgb_top_10


 20%|====                | 511/2500 [00:11<00:42]       

 23%|=====               | 565/2500 [00:12<00:41]       

 25%|=====               | 622/2500 [00:13<00:39]       

 27%|=====               | 681/2500 [00:14<00:37]       

 30%|======              | 739/2500 [00:15<00:35]       

...

 92%|==================  | 2303/2500 [00:42<00:03]       

 94%|=================== | 2353/2500 [00:43<00:02]       

 96%|=================== | 2407/2500 [00:44<00:01]       

 99%|===================| 2466/2500 [00:45<00:00]       

,0,1
0,developed_percent,0.483098
1,population_density,0.425507
2,total_housing,0.312309
3,Solar Radiation 7 Day Avg,0.285942
4,intermix_zone,0.228234
5,NDVI_mean_difference,0.205906
6,interface_zone,0.195931
7,Daily Maximum Air Temperature 7 Day Avg,0.178608
8,Palmer Drought Severity Index,0.166239
9,SPI 180-Day,0.156734


### RF Ignition SHAP

In [9]:
rf_top_10 = get_shap(ignition_rf, X_rf, y_rf)
rf_top_10 

 62%|============        | 1556/2500 [00:11<00:06]       

 68%|==============      | 1709/2500 [00:12<00:05]       

 74%|===============     | 1847/2500 [00:13<00:04]       

 79%|================    | 1975/2500 [00:14<00:03]       

 85%|=================   | 2120/2500 [00:15<00:02]       

 91%|==================  | 2271/2500 [00:16<00:01]       

 97%|=================== | 2426/2500 [00:17<00:00]       

,0,1
0,developed_percent,0.021244
1,total_housing,0.017884
2,influence_zone,0.015622
3,population_density,0.015376
4,median_income,0.013433
5,total_population,0.013127
6,housing_density,0.013097
7,northness_mean,0.011672
8,northness_mean_x_Daily Maximum Air Temperature,0.010303
9,intermix_zone,0.009463


## Set Ablation

In [10]:
for set_name, columns in feature_sets.items():
    print(f"{set_name}: {columns}")
    print("\n")

Water Demand: ['Actual Evapotranspiration', 'Solar Radiation', 'Solar Radiation 7 Day Avg', 'Daily Minimum Air Temperature', 'Daily Maximum Air Temperature', 'Daily Maximum Air Temperature 7 Day Avg', 'Daily Minimum Air Temperature 7 Day Avg', 'Vapor Pressure Deficit', 'Vapor Pressure Deficit 7 Day Avg', 'Wind Speed', 'Wind Speed 7 Day Avg']


Water Supply: ['Precipitation', 'Precipitation 7 Day Avg', 'Maximum Relative Humidity', 'Minimum Relative Humidity', 'Maximum Relative Humidity 7 Day Avg', 'Minimum Relative Humidity 7 Day Avg', 'Specific Humidity', '100-hour Dead Fuel Moisture', '1000-hour Dead Fuel Moisture']


Water Supply Indexes: ['SPI 30-Day', 'SPI 180-Day', 'SPEI 30-Day', 'SPEI 90-Day', 'SPEI 180-Day', 'Palmer Drought Severity Index']


Fire Danger: ['Burning Index', 'Energy Release Component', 'Santa_Ana_Score']


Social: ['total_housing', 'total_population', 'housing_density', 'population_density', 'median_income']


Infrastructure: ['power_line_meters', 'power_line_dens

In [11]:
compare = []

compare.append(compare_model(ignition_xbg, X, y, best_strategy,
                                     name = 'XGB', test_set = 'Full'))

compare.append(compare_model(ignition_rf, X, y, best_strategy,
                                     name = 'RF', test_set = 'Full'))

for set_name, set_list in feature_sets.items():
    for model_name, model in models.items():
        print("Testing " + f"{model_name}: {set_name}")
        X_section = X.drop(columns = set_list)
        compare.append(compare_model(model, X_section, y, best_strategy,
                                     name = model_name, test_set = set_name))

Testing XGB: Water Demand


Testing RF: Water Demand


Testing XGB: Water Supply


Testing RF: Water Supply


Testing XGB: Water Supply Indexes


...

Testing RF: Others


Testing XGB: Coded Ecoregions


Testing RF: Coded Ecoregions


Testing XGB: Coded Seasons


Testing RF: Coded Seasons


In [12]:
comparisons = pd.DataFrame(compare)

In [13]:
XGB = comparisons[comparisons['Model'] == 'XGB'] 
RF = comparisons[comparisons['Model'] == 'RF'] 

In [14]:
full_XGB = XGB.loc[XGB['Test Set']=='Full','Macro F1'].values[0]
full_RF = RF.loc[RF['Test Set']=='Full','Macro F1'].values[0]

In [15]:
XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100

C:\Users\dusti\AppData\Local\Temp\ipykernel_10796\2524239291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  XGB['Macro F1 Percent Difference'] = ((full_XGB - XGB['Macro F1']) / full_XGB) * 100
C:\Users\dusti\AppData\Local\Temp\ipykernel_10796\2524239291.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  RF['Macro F1 Percent Difference'] = ((full_RF - RF['Macro F1']) / full_RF) * 100


In [16]:
RF

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
1,Full,RF,0.467405,0.444616,0.322917,0.000000
3,Water Demand,RF,0.453488,0.432453,0.333333,2.735623
5,Water Supply,RF,0.474357,0.451992,0.364583,-1.659043
7,Water Supply Indexes,RF,0.473375,0.451224,0.333333,-1.486176
9,Fire Danger,RF,0.470498,0.448394,0.354167,-0.849706
11,Social,RF,0.462680,0.441434,0.322917,0.715625
13,Infrastructure,RF,0.473933,0.451201,0.395833,-1.481058
15,Elevation,RF,0.462023,0.439915,0.322917,1.057410
17,WUI,RF,0.476212,0.453914,0.343750,-2.091371
19,Ecoregion,RF,0.465443,0.443156,0.354167,0.328382


In [17]:
XGB

,Test Set,Model,Weighted F1,Macro F1,High Risk Recall,Macro F1 Percent Difference
0,Full,XGB,0.446622,0.424865,0.260417,0.000000
2,Water Demand,XGB,0.444739,0.422260,0.239583,0.612982
4,Water Supply,XGB,0.453244,0.430424,0.291667,-1.308442
6,Water Supply Indexes,XGB,0.457268,0.435989,0.260417,-2.618240
8,Fire Danger,XGB,0.432756,0.411073,0.281250,3.246138
10,Social,XGB,0.438063,0.416048,0.250000,2.075128
12,Infrastructure,XGB,0.438173,0.415979,0.260417,2.091445
14,Elevation,XGB,0.452095,0.429484,0.281250,-1.087263
16,WUI,XGB,0.465917,0.446185,0.322917,-5.018157
18,Ecoregion,XGB,0.446386,0.425649,0.250000,-0.184631
